# Resume–Job Matching Engine: full training on Colab (T4)

This notebook trains both bi-encoder runs on the full data and checks System D. It runs the
repository's own scripts; it contains no training logic of its own.

1. checks the GPU · 2. mounts Drive · 3. clones the repo · 4. installs dependencies (keeps Colab's CUDA torch)
· 5. prepares the data, verifies the **labels-v1** freeze, mines negatives, trains `biencoder_hard` and
`biencoder_random` · 6. builds the fine-tuned FAISS index and runs the real System D test · 7. prints where
everything was saved.

**Before you start**

- Push your latest commits to GitHub (the notebook clones `main`).
- Upload the raw datasets to Google Drive, same layout as on your laptop:
  `MyDrive/rjm/data/raw/resume/Resume.csv` and `MyDrive/rjm/data/raw/linkedin/`
  (`job_postings.csv`, `companies.csv`, `job_skills.csv`). The notebook never downloads datasets.
- Private repo only: add a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar), holding a
  GitHub token with read access to the repo, and allow this notebook to use it. The token is never printed.
- Runtime → Change runtime type → **T4 GPU**.

**Safe to rerun.** Data, negatives, checkpoints, models and indexes live on Drive under `MyDrive/rjm/`.
If the session disconnects, reconnect and choose *Run all* again: finished steps are skipped (each
script checks its inputs' fingerprints), and an interrupted training run resumes from its latest
complete checkpoint (saved every 100 steps). Never add `--force` or `--unfreeze` here.

Drive space: each run keeps up to two checkpoints with optimizer state (about 270 MB each for MiniLM)
plus its final model (about 90 MB), so plan for roughly 1.5 GB.

## Settings

In [ ]:
# --- Settings (edit if needed) ---
REPO_URL = "https://github.com/pariakkaram/resume-job-matching-engine.git"
BRANCH = "main"
DRIVE_MOUNT = "/content/drive"
RJM_ROOT = DRIVE_MOUNT + "/MyDrive/rjm"           # data/, artifacts/, results/ and export/ live here
REPO_DIR = "/content/resume-job-matching-engine"   # code: recloned or updated every session
VENV_DIR = "/content/rjm-venv"                      # project packages, layered over Colab's own (incl. torch)
RUNS = ["biencoder_hard", "biencoder_random"]       # hard first: it is the model Systems C and D use

import hashlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

DATA_DIR = Path(RJM_ROOT) / "data"
ARTIFACTS_DIR = Path(RJM_ROOT) / "artifacts"
RESULTS_DIR = Path(RJM_ROOT) / "results"
VENV_PY = VENV_DIR + "/bin/python"

# Environment for every project command: the storage roots point at Drive (src/config.py).
ENV = {
    **os.environ,
    "RJM_DATA_DIR": str(DATA_DIR),
    "RJM_ARTIFACTS_DIR": str(ARTIFACTS_DIR),
    "RJM_RESULTS_DIR": str(RESULTS_DIR),
    "PYTHONUNBUFFERED": "1",
    "TQDM_MININTERVAL": "20",        # fewer progress-bar lines in the cell output
    "HF_HUB_DISABLE_SYMLINKS_WARNING": "1",
}
ENV.pop("RJM_SMOKE_TEST", None)      # full data only


class StepFailed(RuntimeError):
    pass


def run(args, ok_codes=(0,), cwd=REPO_DIR):
    """Run one command, stream its output, return (exit code, output). Other exit codes raise StepFailed."""
    args = [str(a) for a in args]
    print("$", " ".join(args), flush=True)
    proc = subprocess.Popen(args, cwd=cwd, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    code = proc.wait()
    if code not in ok_codes:
        raise StepFailed(f"exit code {code} from: {' '.join(args)}")
    return code, "".join(lines)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

## 1. Check the GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise StepFailed("No GPU. Runtime > Change runtime type > T4 GPU, then Run all again.")
COLAB_TORCH = torch.__version__
print("GPU:", torch.cuda.get_device_name(0))
print("torch", COLAB_TORCH, "| CUDA", torch.version.cuda)
print("Python", sys.version.split()[0])
if sys.version_info[:2] not in ((3, 10), (3, 11)):
    print("NOTE: the pinned requirements were tested on Python 3.11; if installation fails, the pins "
          "need a Colab-compatible version (see the pip error).")

## 2. Mount Google Drive and check the raw data

In [ ]:
from google.colab import drive

if not os.path.ismount(DRIVE_MOUNT):
    drive.mount(DRIVE_MOUNT)
for directory in (DATA_DIR / "raw", ARTIFACTS_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

REQUIRED_RAW = ["resume/Resume.csv", "linkedin/job_postings.csv", "linkedin/companies.csv", "linkedin/job_skills.csv"]
missing = [name for name in REQUIRED_RAW if not (DATA_DIR / "raw" / name).is_file()]
if missing:
    raise StepFailed(
        "Missing raw files under MyDrive/rjm/data/raw/: " + ", ".join(missing)
        + ". Upload them from your laptop's data/raw/ (same folder layout) and rerun. "
        "All four are needed to rebuild the frozen labels-v1 corpus."
    )
for name in REQUIRED_RAW:
    size = (DATA_DIR / "raw" / name).stat().st_size / 1e6
    print(f"ok  data/raw/{name}  ({size:.1f} MB)")

## 3. Clone or update the repository

In [ ]:
import base64


def _github_token():
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN") or None
    except Exception:  # secret not defined, or notebook access not granted
        return None


TOKEN = _github_token()
_SECRETS = []
if TOKEN:
    _HEADER = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{TOKEN}".encode()).decode()
    _SECRETS = [TOKEN, _HEADER.split()[-1]]


def git(*args, cwd=None):
    """Run git. The token travels in an environment-only HTTP header: it is not on the command line,
    not stored in .git/config and never printed (it is also masked in any error output)."""
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    if TOKEN:
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_HEADER)
    result = subprocess.run(["git", *args], cwd=cwd, env=env, capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    for secret in _SECRETS:
        output = output.replace(secret, "***")
    if result.returncode != 0:
        hint = "" if TOKEN else " If the repository is private, add the GITHUB_TOKEN secret and allow this notebook to use it."
        raise StepFailed(f"git {args[0]} failed:\n{output}\n{hint}")
    return output


print("GitHub token:", "found in Colab secrets (not shown)" if TOKEN else "not set (fine for a public repository)")
if (Path(REPO_DIR) / ".git").is_dir():
    git("checkout", BRANCH, cwd=REPO_DIR)
    git("pull", "--ff-only", "origin", BRANCH, cwd=REPO_DIR)  # fails safely if the clone was edited here
else:
    git("clone", "--branch", BRANCH, REPO_URL, REPO_DIR)
COMMIT = git("rev-parse", "HEAD", cwd=REPO_DIR)
print("Code:", BRANCH, "@", COMMIT)

## 4. Install dependencies (keeping Colab's GPU torch)

In [ ]:
# A virtual environment layered over Colab's packages (--system-site-packages): it sees Colab's CUDA torch,
# and the project's pinned packages are installed into it without touching Colab's own environment.
# torch is removed from requirements.txt and pinned to Colab's version as a constraint, so pip can never
# replace it; if a pinned package needs another torch, pip stops with an error instead.
requirements = (Path(REPO_DIR) / "requirements.txt").read_text(encoding="utf-8").splitlines()
kept = [line for line in requirements if not re.match(r"\s*torch\s*(==|>=|<=|~=|$)", line)]
install_key = hashlib.sha256(("\n".join(kept) + COLAB_TORCH).encode()).hexdigest()
marker = Path(VENV_DIR) / ".rjm-installed"

if marker.is_file() and marker.read_text() == install_key:
    print("Dependencies already installed in this session.")
else:
    if not Path(VENV_PY).exists():
        # --without-pip: Colab's python may lack ensurepip; the system pip installs into the venv anyway.
        run([sys.executable, "-m", "venv", "--system-site-packages", "--without-pip", VENV_DIR], cwd="/content")
    Path("/content/rjm-requirements.txt").write_text("\n".join(kept) + "\n", encoding="utf-8")
    Path("/content/rjm-constraints.txt").write_text(f"torch=={COLAB_TORCH}\n", encoding="utf-8")
    run([VENV_PY, "-m", "pip", "install", "-q", "-r", "/content/rjm-requirements.txt",
         "-c", "/content/rjm-constraints.txt"], cwd="/content")
    marker.write_text(install_key)

_, out = run([VENV_PY, "-c", "import torch, sentence_transformers, transformers, faiss; "
              "print(torch.__version__, torch.cuda.is_available(), sentence_transformers.__version__, transformers.__version__)"])
venv_torch, cuda_ok = out.split()[:2]
if venv_torch != COLAB_TORCH or cuda_ok != "True":
    raise StepFailed(f"The project environment uses torch {venv_torch} (CUDA {cuda_ok}); expected Colab's {COLAB_TORCH} with CUDA.")
print("OK: project environment uses Colab's GPU torch", venv_torch)

## 5a. Prepare the data

Skipped automatically if `MyDrive/rjm/data/processed/` is already up to date. If it stops with
**FROZEN** (exit code 3), the corpus built here differs from labels-v1. Do not unfreeze. Instead, upload
your laptop's `data/processed/` folder to `MyDrive/rjm/data/processed/` and rerun this cell:
`prepare_data` recognises it as up to date for the same raw files and keeps it.

In [ ]:
code, _ = run([VENV_PY, "scripts/prepare_data.py"], ok_codes=(0, 3))
if code == 3:
    raise StepFailed(
        "FROZEN: the corpus or labels.yaml differ from labels-v1 (data/labels/FROZEN.json). "
        "Do not use --unfreeze. Upload your laptop's data/processed/ to MyDrive/rjm/data/processed/ and rerun."
    )

## 5b. Verify the labels-v1 freeze

In [ ]:
frozen = read_json(Path(REPO_DIR) / "data" / "labels" / "FROZEN.json")
manifest = read_json(DATA_DIR / "processed" / "manifest.json")
checks = {
    "corpus fingerprint": (manifest.get("corpus_fingerprint"), frozen["corpus_fingerprint"]),
    "labels.yaml hash": (manifest.get("mapping_version"), frozen["labels_yaml_sha256"]),
}
for name, (current, expected) in checks.items():
    print(f"{'ok ' if current == expected else 'BAD'} {name}: {str(current)[:16]}... (frozen {expected[:16]}...)")
if any(current != expected for current, expected in checks.values()) or manifest.get("freeze_overridden"):
    raise StepFailed("The processed data does not match labels-v1. Stop here and compare with your laptop.")
# The project's own check as well (labels.yaml in this clone and the corpus).
run([VENV_PY, "-c", "import json, sys; from src.data.freeze import check_frozen; "
     f"check_frozen(corpus_fingerprint={manifest['corpus_fingerprint']!r}); print('check_frozen: OK, version', "
     f"{frozen['version']!r})"])

## 5c. Mine the training negatives (random and BM25, training resumes only)

In [ ]:
run([VENV_PY, "scripts/train.py", "--mine-only", "--run", "all"])
for mode in ("random", "hard"):
    summary = read_json(ARTIFACTS_DIR / "negatives" / f"{mode}.meta.json")["summary"]
    print(f"{mode}: {summary['pairs']} pairs from {summary['resumes']} resumes, sources {summary['negative_source']}")

## 5d. Train `biencoder_hard` (the model used by Systems C and D)

In [ ]:
run([VENV_PY, "scripts/train.py", "--run", "biencoder_hard"])

## 5e. Train `biencoder_random` (the random-negative ablation arm)

In [ ]:
run([VENV_PY, "scripts/train.py", "--run", "biencoder_random"])

## 6a. Build the fine-tuned FAISS index (System C; System D reuses it)

In [ ]:
run([VENV_PY, "scripts/build_index.py", "--system", "finetuned"])

## 6b. Run System D and the real-data System D test

The demo query also downloads the cross-encoder. The test is the one skipped on the laptop; this cell
fails unless it actually **ran and passed** (a skip counts as a failure here).

In [ ]:
import xml.etree.ElementTree as ET

_, out = run([VENV_PY, "scripts/search.py", "--system", "finetuned_rerank", "--demo"])
if "FALLBACK" in out:
    raise StepFailed("System D fell back to bi-encoder order: the cross-encoder did not load (see the reason above).")

report = "/content/system_d_test.xml"
run([VENV_PY, "-m", "pytest", "tests/test_reranker.py::test_real_system_d_end_to_end",
     "-m", "integration", "-q", "-p", "no:cacheprovider", f"--junitxml={report}"])
suite = ET.parse(report).getroot()
suite = suite if suite.tag == "testsuite" else suite.find("testsuite")
counts = {k: int(suite.get(k, 0)) for k in ("tests", "failures", "errors", "skipped")}
if counts != {"tests": 1, "failures": 0, "errors": 0, "skipped": 0}:
    raise StepFailed(f"The System D test did not run and pass: {counts}")
print("System D test: 1 passed (not skipped)")

## 7. Where everything was saved

In [ ]:
def size_mb(path):
    path = Path(path)
    files = [path] if path.is_file() else [p for p in path.rglob("*") if p.is_file()]
    return sum(p.stat().st_size for p in files) / 1e6


print("Code:", BRANCH, "@", COMMIT, "\n")
print("Processed data :", DATA_DIR / "processed", f"({size_mb(DATA_DIR / 'processed'):.0f} MB)")
for mode in ("random", "hard"):
    path = ARTIFACTS_DIR / "negatives" / f"{mode}.parquet"
    print(f"Negatives {mode:<6}:", path, f"({size_mb(path):.1f} MB)")
for run_name in RUNS:
    run_dir = ARTIFACTS_DIR / "models" / run_name
    meta = read_json(run_dir / "meta.json")
    print(f"\nModel {run_name}: {run_dir}  ({size_mb(run_dir):.0f} MB incl. checkpoints)")
    print(f"  final model     : {run_dir / 'final'}")
    print(f"  config, log     : {run_dir / 'training_config.yaml'}, {run_dir / 'train_log.jsonl'}")
    print(f"  steps {meta['global_step']}, epochs {meta['epochs_completed']}, train loss {meta['train_loss']:.4f}, "
          f"runtime {meta['train_runtime_seconds']:.0f} s on {meta['environment'].get('gpu')}")
    print(f"  best {meta['metric_for_best_model']} = {meta['best_metric']:.4f} (weak labels, validation resumes)")
    print(f"  best checkpoint : {meta['best_checkpoint']}")
    print(f"  negatives       : {meta['negatives_summary']['negative_source']}")
index = ARTIFACTS_DIR / "indexes" / "finetuned"
print(f"\nFine-tuned index: {index}  ({size_mb(index):.0f} MB), built from {read_json(index / 'meta.json').get('training_run')}")
print("\nValidation NDCG@10 uses weak labels and is for checkpoint selection only; test results come in Phase 6.")

## 8. (Optional) Pack the artifacts for your laptop

Writes `MyDrive/rjm/export/rjm-colab-artifacts.zip` with what the laptop needs: both final models with
their `meta.json`, configs and logs (no checkpoints), the negatives and the fine-tuned index. Rerunning
replaces the zip. See *Copying Colab artifacts back to the laptop* in the README.

In [ ]:
import zipfile

export_dir = Path(RJM_ROOT) / "export"
export_dir.mkdir(parents=True, exist_ok=True)
target = export_dir / "rjm-colab-artifacts.zip"
tmp = export_dir / "rjm-colab-artifacts.zip.tmp"   # renamed only when complete
members = []
for run_name in RUNS:
    run_dir = ARTIFACTS_DIR / "models" / run_name
    members += [p for p in (run_dir / "final").rglob("*") if p.is_file()]
    members += [run_dir / n for n in ("meta.json", "run_inputs.json", "training_config.yaml", "train_log.jsonl")]
members += sorted((ARTIFACTS_DIR / "negatives").glob("*"))
members += [p for p in (ARTIFACTS_DIR / "indexes" / "finetuned").rglob("*") if p.is_file()]
with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in members:
        archive.write(path, Path("artifacts") / path.relative_to(ARTIFACTS_DIR))
os.replace(tmp, target)
print(f"Wrote {target} ({target.stat().st_size / 1e6:.0f} MB, {len(members)} files)")